<a href="https://colab.research.google.com/github/sde2424242424-coder/2026_spring_assignments/blob/main/CountVectorizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP 기초: CountVectorizer를 이용한 뉴스 그룹 대화 분류(코사인유사성)

In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

## [Step 1] 데이터 로드 및 샘플링

In [2]:
categories = ['comp.graphics', 'sci.space', 'talk.religion.misc']

newsgroups = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes')
)

data, labels = [], []

samples_per_category = 20  # 기본 요구사항: 각 주제당 20개

for i, category in enumerate(categories):
    indices = np.where(newsgroups.target == i)[0][:samples_per_category]
    for idx in indices:
        data.append(newsgroups.data[idx])
        labels.append(newsgroups.target_names[i])

print("총 문서 수:", len(data))
print("라벨 예시:", labels[:5])

총 문서 수: 60
라벨 예시: ['comp.graphics', 'comp.graphics', 'comp.graphics', 'comp.graphics', 'comp.graphics']


## [Step 2] CountVectorizer 설정 및 변환

In [3]:
vectorizer = CountVectorizer(stop_words='english')
count_matrix = vectorizer.fit_transform(data)

print("벡터 차원:", count_matrix.shape)
print("일부 feature:", vectorizer.get_feature_names_out()[:20])

벡터 차원: (60, 2342)
일부 feature: ['000' '02' '0865' '10' '100' '10bps' '11' '111' '12' '13' '130' '131'
 '14' '142' '15' '150' '1500' '152' '15rpm' '16']


## [Step 3] 분류 함수 구현



In [4]:
def classify_text(input_text):
    input_vec = vectorizer.transform([input_text])
    sim = cosine_similarity(input_vec, count_matrix)
    best_idx = np.argmax(sim)
    return labels[best_idx], sim[0][best_idx]

유사도가 0.0000이라는 것은 입력 문장에 있는 단어들이 학습 데이터의 단어 사전에 거의 없다는 의미입니다. 즉, CountVectorizer로 변환했을 때 입력 문장 벡터와 학습 문서 벡터 사이에 공통 단어가 없어서 코사인 유사도가 0이 됩니다. 그런데도 분류 결과가 나오는 이유는 np.argmax()가 가장 큰 값을 가진 인덱스를 반환하기 때문입니다. 모든 유사도 값이 0이어도 첫 번째 최대값의 인덱스를 반환하므로 특정 카테고리가 선택됩니다. 하지만 이 경우는 실제로 의미 있는 분류라고 보기 어렵습니다.

## [Step 4] 테스트 실행

In [5]:
test_sentences = [
    "The rocket launched into orbit.",
    "A new 3D rendering technique for graphics.",
    "Theological discussions on faith and god.",
    "Exploring the mars with a robotic rover."
]

print("\n[기본 CountVectorizer 결과]")
for s in test_sentences:
    cat, score = classify_text(s)
    print(f"문장: {s}")
    print(f"예측 주제: {cat} | 유사도: {score:.4f}")
    print("-" * 50)



[기본 CountVectorizer 결과]
문장: The rocket launched into orbit.
예측 주제: sci.space | 유사도: 0.0870
--------------------------------------------------
문장: A new 3D rendering technique for graphics.
예측 주제: comp.graphics | 유사도: 0.1952
--------------------------------------------------
문장: Theological discussions on faith and god.
예측 주제: talk.religion.misc | 유사도: 0.3430
--------------------------------------------------
문장: Exploring the mars with a robotic rover.
예측 주제: comp.graphics | 유사도: 0.0000
--------------------------------------------------


## [Step 5] 모든 문서와의 유사도 확인 함수

In [6]:
def classify_text_with_detail(input_text):
    input_vec = vectorizer.transform([input_text])
    sim = cosine_similarity(input_vec, count_matrix)[0]
    best_idx = np.argmax(sim)
    return labels[best_idx], sim[best_idx], sim

example_text = "Exploring the mars with a robotic rover."
cat, score, all_scores = classify_text_with_detail(example_text)

print("\n[상세 분석]")
print("입력 문장:", example_text)
print("예측 주제:", cat)
print("최대 유사도:", round(score, 4))
print("전체 유사도 중 상위 5개:", sorted(all_scores, reverse=True)[:5])


[상세 분석]
입력 문장: Exploring the mars with a robotic rover.
예측 주제: comp.graphics
최대 유사도: 0.0
전체 유사도 중 상위 5개: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]


## [Step 6] 성능 개선 실험 1: 샘플 수 증가

In [7]:
def run_experiment_countvectorizer(samples_per_category=20):
    exp_data, exp_labels = [], []

    for i, category in enumerate(categories):
        indices = np.where(newsgroups.target == i)[0][:samples_per_category]
        for idx in indices:
            exp_data.append(newsgroups.data[idx])
            exp_labels.append(newsgroups.target_names[i])

    exp_vectorizer = CountVectorizer(stop_words='english')
    exp_matrix = exp_vectorizer.fit_transform(exp_data)

    print(f"\n[실험] CountVectorizer | 주제당 샘플 수: {samples_per_category}")
    for s in test_sentences:
        input_vec = exp_vectorizer.transform([s])
        sim = cosine_similarity(input_vec, exp_matrix)
        best_idx = np.argmax(sim)
        print(f"문장: {s}")
        print(f"예측 주제: {exp_labels[best_idx]} | 유사도: {sim[0][best_idx]:.4f}")
        print("-" * 50)

run_experiment_countvectorizer(samples_per_category=100)


[실험] CountVectorizer | 주제당 샘플 수: 100
문장: The rocket launched into orbit.
예측 주제: sci.space | 유사도: 0.2858
--------------------------------------------------
문장: A new 3D rendering technique for graphics.
예측 주제: comp.graphics | 유사도: 0.3097
--------------------------------------------------
문장: Theological discussions on faith and god.
예측 주제: talk.religion.misc | 유사도: 0.3357
--------------------------------------------------
문장: Exploring the mars with a robotic rover.
예측 주제: sci.space | 유사도: 0.1041
--------------------------------------------------


주제별 샘플 수를 20개에서 100개로 늘리면 학습 데이터의 단어 종류가 더 다양해집니다. 따라서 입력 문장과 겹치는 단어가 나올 가능성이 높아지고, 유사도 점수도 더 안정적으로 증가합니다. 결과적으로 분류 정확도가 개선될 수 있습니다.

## [Step 7] 성능 개선 실험 2: TfidfVectorizer

In [8]:
def run_experiment_tfidf(samples_per_category=20):
    exp_data, exp_labels = [], []

    for i, category in enumerate(categories):
        indices = np.where(newsgroups.target == i)[0][:samples_per_category]
        for idx in indices:
            exp_data.append(newsgroups.data[idx])
            exp_labels.append(newsgroups.target_names[i])

    tfidf_vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf_vectorizer.fit_transform(exp_data)

    print(f"\n[실험] TfidfVectorizer | 주제당 샘플 수: {samples_per_category}")
    for s in test_sentences:
        input_vec = tfidf_vectorizer.transform([s])
        sim = cosine_similarity(input_vec, tfidf_matrix)
        best_idx = np.argmax(sim)
        print(f"문장: {s}")
        print(f"예측 주제: {exp_labels[best_idx]} | 유사도: {sim[0][best_idx]:.4f}")
        print("-" * 50)

run_experiment_tfidf(samples_per_category=20)


[실험] TfidfVectorizer | 주제당 샘플 수: 20
문장: The rocket launched into orbit.
예측 주제: sci.space | 유사도: 0.0890
--------------------------------------------------
문장: A new 3D rendering technique for graphics.
예측 주제: comp.graphics | 유사도: 0.1689
--------------------------------------------------
문장: Theological discussions on faith and god.
예측 주제: talk.religion.misc | 유사도: 0.3086
--------------------------------------------------
문장: Exploring the mars with a robotic rover.
예측 주제: comp.graphics | 유사도: 0.0000
--------------------------------------------------


TfidfVectorizer는 단순 단어 빈도뿐 아니라 단어의 중요도까지 반영합니다.
여러 문서에서 너무 자주 등장하는 일반 단어의 가중치는 낮추고, 특정 문서에서 중요한 단어의 가중치는 높입니다.
그래서 보통 CountVectorizer보다 더 나은 분류 성능을 보여줄 수 있습니다.

## [Step 8] 성능 개선 실험 3: ngram_range=(1, 2)

In [9]:
def run_experiment_ngram(samples_per_category=20):
    exp_data, exp_labels = [], []

    for i, category in enumerate(categories):
        indices = np.where(newsgroups.target == i)[0][:samples_per_category]
        for idx in indices:
            exp_data.append(newsgroups.data[idx])
            exp_labels.append(newsgroups.target_names[i])

    ngram_vectorizer = CountVectorizer(stop_words='english', ngram_range=(1, 2))
    ngram_matrix = ngram_vectorizer.fit_transform(exp_data)

    print(f"\n[실험] CountVectorizer + ngram_range=(1,2) | 주제당 샘플 수: {samples_per_category}")
    for s in test_sentences:
        input_vec = ngram_vectorizer.transform([s])
        sim = cosine_similarity(input_vec, ngram_matrix)
        best_idx = np.argmax(sim)
        print(f"문장: {s}")
        print(f"예측 주제: {exp_labels[best_idx]} | 유사도: {sim[0][best_idx]:.4f}")
        print("-" * 50)

run_experiment_ngram(samples_per_category=20)


[실험] CountVectorizer + ngram_range=(1,2) | 주제당 샘플 수: 20
문장: The rocket launched into orbit.
예측 주제: sci.space | 유사도: 0.0685
--------------------------------------------------
문장: A new 3D rendering technique for graphics.
예측 주제: comp.graphics | 유사도: 0.1469
--------------------------------------------------
문장: Theological discussions on faith and god.
예측 주제: talk.religion.misc | 유사도: 0.2462
--------------------------------------------------
문장: Exploring the mars with a robotic rover.
예측 주제: comp.graphics | 유사도: 0.0000
--------------------------------------------------


ngram_range=(1,2)를 설정하면 단일 단어뿐 아니라 두 단어 묶음도 특징으로 사용합니다.
예를 들어 “space shuttle”, “3D rendering” 같은 표현을 더 잘 반영할 수 있어서 문맥 정보가 향상됩니다.
하지만 feature 수가 크게 늘어나므로 벡터 차원이 커지고 계산량도 증가합니다.